In [1]:
import pandas as pd
fp = "data/coffee_prices.csv"

In [2]:
df = pd.read_csv(fp)

In [3]:
df.columns = ["date", "cents-per-lb"]
df.date = pd.to_datetime(df.date)

In [4]:
price_series = df["cents-per-lb"]
price_series.index = df["date"]

In [5]:
from scipy.stats import gaussian_kde
import numpy as np
kde = gaussian_kde(price_series)


# Retrieve the calculated bandwidth (it's the covariance_factor * standard deviation)
# The covariance_factor is a scaling coefficient
scott_factor = kde.covariance_factor()
bw = scott_factor * np.std(price_series, ddof=1)

In [6]:
class TSSlidingWindowDataset:

    def __init__(self, window_size, series):
        self._ws = window_size
        self._series = series
        self._series = self._series.reset_index(drop=True)
        self._qbottom = -1
        self._windowdf = pd.DataFrame()
        self._resp_idx = -1 

    def initialize_window(self):
        start_idx = 0
        end_idx = start_idx + (self._ws -1)
        
        for index, value in self._series.loc[start_idx:end_idx].items():
            self._windowdf.loc[0, index] = value
    
        self._windowdf.columns = ["w-" + str(i + 1) for i in range(self._ws)]
        self._windowdf["resp"] = self._series.loc[end_idx + 1]
        self._resp_idx = end_idx + 1
        return

    def enque(self):
        last_row = self._windowdf.shape[0] - 1

        new_data = self._windowdf.iloc[last_row,1:].tolist()
    
        cols = ["w-" + str(i + 1) for i in range(self._ws)]
        new_df = pd.DataFrame([new_data], columns=cols)
        
        self._resp_idx += 1
        new_df["resp"] = self._series.loc[self._resp_idx]
        self._windowdf = pd.concat([self._windowdf, new_df])

    def create_windowed_dataset(self):
        self.initialize_window()
        
        eos_index = self._series.shape[0] - 1

        while self._resp_idx < eos_index:
            self.enque()

        return self._windowdf
        
        
        
        
        
        
        

In [7]:
tsw = TSSlidingWindowDataset(5, price_series)

In [8]:
df = tsw.create_windowed_dataset()

In [14]:
df.corrwith(axis=1)

TypeError: DataFrame.corrwith() missing 1 required positional argument: 'other'

In [10]:
cols = ["w-" + str(i+1) for i in range(5)]
exog_data = df[cols].values
endog_data = df["resp"].values


In [13]:
import statsmodels.api as sm
import numpy as np



# 2. Initialize the KernelReg class
# 'c' denotes a continuous variable type
# bw='cv_ls' uses least-squares cross-validation for bandwidth selection
# ckertype='gaussian' specifies the kernel function
kreg = sm.nonparametric.KernelReg(
    endog=endog_data,
    exog=exog_data,
    var_type='ccccc',
    bw='aic',
    ckertype='gaussian'
)

# 3. Fit the model and get the results (mean prediction)
# data_predict=exog_data calculates the results at the original data points
mean_prediction, marginal_effects = kreg.fit(data_predict=exog_data)

# 4. Access other results, like R-squared
r_squared = kreg.r_squared

print(f"R-squared: {r_squared}")

R-squared: <bound method KernelReg.r_squared of KernelReg instance
Number of variables: k_vars = 5
Number of samples:   N = 421
Variable types:      ccccc
BW selection method: aic
Estimator type: ll
>
